# Gaussian Noise 추가

원본 multi-echo GRE complex 데이터에 Gaussian noise를 추가하여 10%, 30%, 50% noise level의 실험 데이터를 생성한다.

원본 데이터는 그대로 유지하고, noise가 추가된 데이터만 별도의 `.mat` 파일로 저장한다.

생성한 noisy 데이터는 이후 Magnitude MP-PCA, Real/Imaginary MP-PCA 등 denoising 실험의 입력 데이터로 사용한다.

In [1]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

### 환경설정

Gaussian noise 생성에 사용할 기본 조건을 설정한다.

- 원본 데이터 경로: `./data/meas_gre_dir1.mat`
- Noise level: 10%, 30%, 50%
- Random seed: 50
- 결과 저장 경로: `./data/noisy_data`

Random seed를 고정하여 실험을 다시 실행하더라도 동일한 random noise pattern을 재현할 수 있도록 한다.

각 noise level에서는 동일한 seed를 사용하고 noise의 크기만 다르게 적용한다.

### 1. 데이터 불러오기

원본 `.mat` 파일에서 noise 생성에 필요한 MRI 데이터를 불러온다.

주요 데이터는 다음과 같다.

- `meas_gre`
  - 원본 multi-echo GRE complex 데이터
  - 실수부와 허수부를 모두 포함한다.
  - 데이터의 마지막 차원은 echo를 의미한다.

- `mask_brain`
  - 뇌 영역을 나타내는 brain mask
  - 뇌 영역은 1, 외부 영역은 0으로 구성된다.

불러온 원본 complex 데이터는 `orig_complex`로 사용하고, brain mask는 boolean 형태로 변환하여 사용한다.

데이터를 불러온 후에는 다음을 확인한다.

- 원본 complex 데이터의 shape
- brain mask의 shape
- echo 개수

본 데이터의 경우 총 6개의 echo가 포함되어 있다.

### 2. Brain mask 내부 magnitude 평균 계산

Gaussian noise의 크기를 결정하기 위한 기준 신호값을 계산한다.

각 echo의 complex 데이터에서 magnitude를 계산한 뒤, brain mask 내부의 voxel만 선택하여 평균 신호값을 구한다.

진행 순서는 다음과 같다.

1. 각 echo의 complex 데이터에서 magnitude 계산
2. Brain mask 내부 voxel만 선택
3. Echo별 magnitude 평균 계산
4. 모든 echo의 평균값을 다시 평균하여 `overall_mean` 계산

`overall_mean`은 이후 10%, 30%, 50% noise의 크기를 결정하는 기준값으로 사용한다.

이 과정에서 magnitude는 noise 크기를 계산하기 위한 용도로만 사용하며, 원본 complex 데이터 자체는 변경하지 않는다.

### 3. 10%, 30%, 50% Gaussian Noise 생성

앞 단계에서 계산한 `overall_mean`을 기준으로 10%, 30%, 50% 수준의 Gaussian noise를 생성한다.

각 noise level의 표준편차 `sigma`는 다음과 같이 설정한다.

- 10%: `overall_mean`의 10%
- 30%: `overall_mean`의 30%
- 50%: `overall_mean`의 50%

각 echo의 complex 데이터를 실수부와 허수부로 분리한 뒤, Real과 Imaginary 각각에 Gaussian noise를 독립적으로 추가한다.

Noise는 brain mask 내부에만 적용하고 mask 외부에는 추가하지 않는다.

Noise가 적용된 실수부와 허수부를 다시 결합하여 `noisy_complex` 데이터를 생성한다.

각 noise level마다 동일한 seed를 다시 사용하기 때문에 10%, 30%, 50%에서 random noise의 pattern은 동일하고 noise의 크기만 달라진다.

생성되는 주요 데이터는 다음과 같다.

- `noisy_real`
- `noisy_imag`
- `noisy_complex`

### 4. Noise 데이터 저장

생성한 10%, 30%, 50% noisy 데이터를 각각 별도의 `.mat` 파일로 저장한다.

저장 파일은 다음과 같다.

- `noisy_meas_gre_dir1_10.mat`
- `noisy_meas_gre_dir1_30.mat`
- `noisy_meas_gre_dir1_50.mat`

각 파일에는 다음 데이터를 저장한다.

#### Noisy 데이터

- `noisy_real`
- `noisy_imag`
- `noisy_complex`

#### Noise 생성 조건

- `noise_ratio`
- `sigma`
- `seed`
- `overall_mean`

#### MRI 정보

- `mask_brain`
- `te_gre`
- `delta_TE`
- `voxel_size_gre`
- `B0_dir`
- `CF`

원본 `meas_gre` 데이터는 기존 원본 파일에 별도로 존재하기 때문에 noisy 결과 파일에는 중복 저장하지 않는다.

### 5. Noise 추가 결과 시각화

Gaussian noise가 정상적으로 추가되었는지 확인하기 위해 원본 영상과 noisy 영상을 비교한다.

본 실험에서는 예시로 다음 위치를 사용한다.

- Echo: 1
- Slice: 88

10%, 30%, 50% 각각에 대해 다음 영상을 확인한다.

- Original Magnitude
  - Noise 추가 전 원본 magnitude 영상

- Noisy Magnitude
  - Gaussian noise가 추가된 magnitude 영상

- Difference
  - Noisy 영상과 Original 영상의 차이

- Brain Mask
  - 실제 noise가 적용된 brain mask 영역

10% → 30% → 50% 순으로 noise가 점차 증가하는지 확인하고, noise가 brain mask 내부에만 적용되었는지도 함께 확인한다.

### 최종 확인사항

Gaussian noise 생성이 완료된 후 다음 사항을 확인한다.

- 원본 complex 데이터가 정상적으로 불러와졌는지 확인
- Brain mask가 정상적으로 불러와졌는지 확인
- Echo별 magnitude 평균이 정상적으로 계산되었는지 확인
- `overall_mean`이 정상적으로 계산되었는지 확인
- 10%, 30%, 50%의 sigma 값이 정상적으로 증가하는지 확인
- Real과 Imaginary 각각에 Gaussian noise가 추가되었는지 확인
- Noise가 brain mask 내부에만 적용되었는지 확인
- 10%, 30%, 50% noisy 파일이 모두 저장되었는지 확인
- 시각화 결과에서 noise level이 증가할수록 noise가 증가하는지 확인

생성된 noisy 데이터는 이후 각 MP-PCA denoising 방식의 입력 데이터로 사용한다.